<a href="https://colab.research.google.com/github/robertbarcik/MCP-tutorial/blob/main/MCP_course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. Welcome: One Server, Any Agent

This is the notebook of the **Model Context Protocol (MCP)** course. You read it top to bottom and run the cells. By the end you have seen every piece of MCP at work: a server, a client, the messages between them, a model driving the server, and a real application using it.

The notebook follows the videos. Chapters are numbered 0 to 4, and every numbered section (0.1, 0.2, ...) is one video. Sections marked *Optional* have no video.

**What we assume you can already do:** write a small Python function, read a dict, and you have once let a model call a function you wrote (the function-calling loop from the previous course). Nothing else.

## 0.1 What MCP is and where it sits

You wrote `search_tickets()` for your IT help desk. It works. Now three people want it: a colleague who lives in Claude Desktop, the team's own agent, and you inside Claude Code. Each of them would need its own glue code, and each time you change the function you would fix three integrations.

**MCP** is an agreed way for a program that owns data or functions to describe them and serve them to any AI application. Write the glue once, and every application that speaks MCP can use it.

Two words we will use all the time, one plain sentence each:

- A **server** is the program that owns the functions and waits to be asked. It does nothing on its own.
- A **client** is the program that asks: Claude Code, an agent framework, or a script you write yourself.

The words say nothing about where the programs run. In this course both run on your machine.

### Where MCP sits

Every AI application that does something in the world has the same four layers. MCP is one of them, and knowing which one keeps the rest of the course in place.

```text
┌─────────────────────────────────────────────────────┐
│  ORCHESTRATION LAYER                                │
│  "When do we call what, with what state,            │
│   under what guardrails, across how many agents?"   │
│                                                     │
│  ADK · LangGraph · CrewAI · OpenAI Agents SDK ·     │
│  Claude Agent SDK · AWS Strands · Azure AI Foundry  │
├─────────────────────────────────────────────────────┤
│  LLM LAYER                                          │
│  The reasoning engine that generates tool calls,    │
│  plans, and responses.                              │
│                                                     │
│  Gemini · Claude · GPT · Mistral · Llama · ...      │
├─────────────────────────────────────────────────────┤
│  TOOL/AGENT ACCESS LAYER                            │
│  How the LLM reaches tools, data, and other agents. │
│                                                     │
│  MCP (tools & data) · A2A (agent-to-agent) ·        │
│  Native function calling · Direct SDK calls         │
├─────────────────────────────────────────────────────┤
│  ACTUAL SYSTEMS                                     │
│  Databases, APIs, files, SaaS apps, other agents    │
└─────────────────────────────────────────────────────┘
```

*(From the publication "Agent Horizon", publications.barcik.training.)*

Read it bottom up. At the bottom are the systems that already exist: your ticket database, a file share, GitHub. At the top, a framework decides when to call what. In between, the model reasons and asks for tools. **MCP lives in the third layer.** It is one answer to a narrow question: how does a model reach a tool that somebody else wrote and runs? Native function calling, which you used in the previous course, is the same layer without a protocol: the function lives inside your own program. The ADK course, which follows this one, is the top layer.

Two consequences. First, MCP does not think, plan or remember; the model and the framework above it do. A server is a menu of functions and the code behind them, nothing more. Second, whatever changes above and below, the questions in this layer stay the same: *what can you do?* and *do this for me.* That is why one hour on MCP is enough, and why this course stays with those two questions rather than with any one version of any one tool.


## 0.2 Materials and setup

Everything for this course is in one folder: this notebook, the `servers/` and `client/` folders with the code we read together, and a README that tells the same story as a text you can re-read later. The same files are on GitHub: https://github.com/robertbarcik/MCP-tutorial. All materials are in English.

### How to read this notebook: two paths

MCP is used by two kinds of people. The colleague who plugs a server into Claude Desktop, judges whether it is any good, and configures it for the team. And the developer who writes servers. This notebook serves both.

- **Everything you see by default is for everyone.** Short cells, plain words, and every cell runs with the Python you already have.
- **🛠️ Developer corner** sections sit at the end of some sections and start collapsed. They hold the proof and the plumbing: what really goes over the wire, why an exception loses your message, how the HTTP server is started. They have no video. Open them when you write code for a living. Nothing outside a corner depends on anything inside it, and *Run all* still runs them.
- **🎯 Mini-task** is for everyone. **🛠️ Stretch** inside a corner is for developers.

If you are not a developer, your goal is concrete: call a tool from Python, read a server file and find its four parts, watch a model drive five servers, and plug a server into a real application by editing one JSON file.

### Setup

Same ritual as the previous course: install, get the repository files (Colab does it for you), load your key. Just run the three cells.

In [1]:
%pip install -q --disable-pip-version-check mcp==2.2.0 openai==3.13.0
print("Packages installed.")

Note: you may need to restart the kernel to use updated packages.
Packages installed.


In [2]:
import os

if "COLAB_GPU" in os.environ or "COLAB_JUPYTER_IP" in os.environ:
    !git clone --depth=1 -q https://github.com/robertbarcik/MCP-tutorial /content/MCP-tutorial 2>/dev/null || true
    os.chdir("/content/MCP-tutorial")
    print(f"Colab: working in {os.getcwd()}")
else:
    print(f"Local: working in {os.getcwd()}")   # run Jupyter from the repository folder

assert os.path.exists("servers/ticket_server.py"), "servers/ folder not found: are you in the repository folder?"
print("servers/ folder found.")

Local: working in /Users/robertbarcik/git-repos/MCP-tutorial
servers/ folder found.


In [3]:
import json, logging, sys
from openai import OpenAI

for name in ("httpx", "httpx2", "openai"):          # otherwise every HTTP request is logged into the cell output
    logging.getLogger(name).setLevel(logging.WARNING)

# Your key, looked up in this order: Colab secret -> environment variable -> a prompt.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
    if not isinstance(api_key, str):                # right after you click "Grant access", Colab can hand back
        api_key = userdata.get("OPENAI_API_KEY")    # something that is not text; asking a second time gives the key
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not isinstance(api_key, str) or not api_key.strip():
    from getpass import getpass
    api_key = getpass("OpenAI API key: ")

api_key = api_key.strip()                           # a pasted key often ends with a newline; the API rejects that header
llm = OpenAI(api_key=api_key)
MODEL = "gpt-5.6-luna"   # small, cheap model of the current generation

try:                                                # Colab plumbing: starting another program needs a real error stream
    sys.stderr.fileno()                             # (explained in the Developer corner of section 1.5)
except Exception:
    sys.stderr = open(os.devnull, "w")

print("Ready. Model:", MODEL)

Ready. Model: gpt-5.6-luna


Three things happened. The two packages are pinned to the same versions as the repository's `requirements.txt`, so Colab and your own machine behave the same. The repository gives this notebook the `servers/` folder, which is where the interesting code lives. And the key was found in Colab secrets, in your environment, or you typed it. That is all the setup there is.

## 0.3 When to use MCP, when not, and our help-desk story

MCP is not the answer to everything. Before we build anything, let's settle when it is worth it.

**Use MCP when the one who calls your functions is not you.** At least one of these is true:

- **You do not control the application.** Colleagues on Claude.ai, ChatGPT or Copilot cannot run your script, but they can connect to your server.
- **The user should not have a shell.** A company wants its ticket system reachable by agents, but only through five vetted functions, with a login, and with a log of every call.
- **Many people share one server.** It runs somewhere central, and nobody installs anything.
- **Your users arrive through an agent.** More and more people ask an assistant to do the thing instead of opening your product themselves. A server is the door that assistant can use.

**Skip MCP when both sides are yours.** If the consumer is a coding agent on your own machine, a command-line tool with good documentation is usually enough. The agent reads the help text, runs the command, and reads the output. Wrapping it in a server adds a layer and takes nothing away.

A good rule of thumb: MCP is for the boundary between you and someone else's agent.

### One story for the whole course: the IT help desk

A support company has five systems, each behind its own small server: **tickets**, **customers**, **billing**, a **knowledge base** of fixes, and **assets** (laptops, servers, warranties). An AI assistant should answer questions like *"which customers have open tickets and overdue invoices?"* by asking the right servers.

```
   Claude Code / your script / an agent framework      <- clients
                        |
        +---------+-----+-----+---------+
        |         |           |         |
    tickets   customers   billing   knowledge base   assets      <- MCP servers
```

The plan, chapter by chapter. Chapter 1: the finished thing first, then the two programs behind it, a server and a client. Chapter 2: a model drives the server. Chapter 3: the server goes into a real application, and we install someone else's. Chapter 4: resources and prompts, and your own task.

Cost: the whole notebook makes a handful of small model calls, a few cents on the course model.

# 1. A Server and a Client

Five short sections. We start with the finished result and then take it apart: the plain function underneath, the server file, the client that asks, and the messages between the two.

## 1.1 Result first: five servers, one chat

Before we take anything apart, see the finished thing. The help desk is five servers and twenty-one tools. You ask a question in plain words; the model picks the tools; a small program keeps five servers running and sends each call to the one that owns the tool. The repository has this as a terminal program, `client/interactive_client.py`. You do not need to read it to use it. By the end of the notebook you will have met every piece of it: the server, the client, the loop, and the routing.

### Run it in a terminal

**In Colab:** open the terminal (`>_` at the bottom left; the packages from the Setup cell are already installed there) and type:

```bash
cd /content/MCP-tutorial
export OPENAI_API_KEY="sk-..."
python client/interactive_client.py
```

**On your own machine:**

```bash
cd MCP-tutorial
source .venv/bin/activate            # Windows: .venv\Scripts\activate
export OPENAI_API_KEY="sk-..."       # Windows: set OPENAI_API_KEY=sk-...
python client/interactive_client.py
```

Questions to try:

- What are all the critical priority tickets?
- Show me customer CUST-001's SLA terms and contacts.
- Which assets have warranties expiring in the next 30 days?
- Which customers have both open tickets and overdue invoices?
- Find similar tickets to TKT-1001 and a knowledge base article that could help.

Type `exit` to stop. All five servers stop with it.

### 🔍 What just happened?

Watch the `->` lines: each one names the tool and, through it, the server that answered. A question like *"open tickets and overdue invoices"* makes the model call the ticket server, then the billing server, then combine. Nobody programmed that route. The rest of the notebook shows how this works, one piece per section, until you could write this program yourself.

### Why this is worth building

The five servers know nothing about this chat program. Tomorrow the same five servers can sit behind Claude Desktop for a colleague, or behind an agent that works through the ticket queue at night with no person typing at all. You build the server once. Who asks, a human in a chat or an agent, can change later.

### 🎯 Mini-task

Ask a question that needs three servers, for example: *"For customer CUST-002: open tickets, outstanding balance, and any asset with an expired warranty."* Count the `->` lines.

## 🛠️ Developer corner: how five servers stay open

Two things in `client/interactive_client.py` are new compared to the client cells later in this notebook:

- `AsyncExitStack` is one `with` block that keeps five `with` blocks open. Without it we would nest five `async with Client(...)` lines.
- `Toolbox` (in `client/agent_loop.py`) remembers which client owns which tool name, so `toolbox.call("get_invoice", ...)` goes to the billing server.

The loop itself is `run_with_tools` from `client/agent_loop.py`, the same loop as in section 2.1, with the routing added.

**🛠️ Stretch.** Add `"hr"` to the `SERVERS` list in `client/interactive_client.py` and ask about the remote work policy. The HR server publishes it as a resource, not a tool. Can this client reach it? Why not, and which of the three MCP things (tool, resource, prompt) does a plain agent loop see at all?

## 1.2 The thing underneath: plain functions

The chat looked clever. Underneath it are ordinary Python functions, and before any protocol we touch one. The ticket server is an ordinary Python file. We import it like any module and call a function. If MCP disappeared tomorrow, this cell would still work.

In [4]:
from servers.ticket_server import TICKETS, search_tickets, get_ticket_details

print(f"The 'database' is a list of {len(TICKETS)} dicts. The first one is a {type(TICKETS[0]).__name__}.\n")

critical = search_tickets(priority="critical")
for ticket in critical["tickets"]:
    print(f"{ticket['ticket_id']}  {ticket['status']:12s} {ticket['subject']}")

The 'database' is a list of 15 dicts. The first one is a dict.

TKT-1002  in_progress  Linux server disk full - /var/log consuming 95% space
TKT-1009  in_progress  Windows 11 BitLocker recovery key prompt on every boot
TKT-1011  resolved     Windows Server 2019 Active Directory replication failing


In [5]:
from pprint import pprint

pprint(get_ticket_details("TKT-9999"))

{'error': 'Ticket TKT-9999 not found',
 'follow_up_tools': ['search_tickets'],
 'reason': 'The ticket_id did not match any tickets in the dataset.',
 'retryable': True,
 'suggested_actions': ['Call search_tickets with a query, customer_id or '
                       'priority filter to rediscover the ticket.',
                       'Verify the ticket_id format (e.g., TKT-1001).'],
 'ticket_id': 'TKT-9999'}


### 🔍 What just happened?

A plain function call, no server anywhere. The second call asked for a ticket that does not exist, and look at what came back: not a crash, but a dict with `error`, `reason`, `suggested_actions` and `follow_up_tools`.

That is a design decision we come back to in section 2.2: **errors are written for the model, not for a human.** A model that reads *"call search_tickets with a query"* knows what to do next. A model that gets a Python traceback does not. Keep this dict in mind; we come back to it when a model is driving.

### The function could be anything

Our function searches a list of tickets. Put something else inside it and nothing else in this course changes. The function could read a mailbox, run a query on a company database, move an object in Blender, or create a design in Canva. Servers for all of these exist: Google publishes one for Gmail, Canva has an official one, and Blender has a popular one written by its community.

Notice what kind of software this is. It is ordinary, "old" software: the same input gives the same output, and there is no AI inside. MCP adds no intelligence to it. MCP only makes it reachable for a model.

This is also a change in how products are built. For years many products had one door: a screen with buttons, made for a person. Now more and more users send an agent instead of clicking themselves, and an agent works far better with functions it can call than with buttons made for a human. So companies add a second door next to the screen: an API, and on top of it an MCP server.

## 1.3 Anatomy of a server

We called the function directly. Now we look at the whole file it lives in, because that file *is* the server.

Open `servers/ticket_server.py` in the file browser on the left. It has four parts, in this order:

1. **Data.** `TICKETS`, the list of dicts you just saw. A real server would read a database; ours keeps a list.
2. **Private helpers.** Small functions the tools use. Their names start with an underscore, which is how we say "not a tool".
3. **Tools.** Plain Python functions with type hints and a docstring: `search_tickets`, `get_ticket_details`, and three more. **The docstring is what the model will read**, so it is written for the model: what the tool does, what each argument means, an example value.
4. **The MCP layer.** A few lines at the very bottom. The only part of the file that knows MCP exists.

The next cell prints that fourth part straight from the file. Three lines matter:

- `mcp = MCPServer("tickets")` gives the server a name.
- `mcp.tool(annotations=READ_ONLY)(search_tickets)` hands an ordinary function to the server. The name, the description and the list of arguments are read from the function itself. You write nothing by hand.
- `mcp.run()` waits for a client.

In [6]:
from pathlib import Path

source = Path("servers/ticket_server.py").read_text()
print(source[source.index("# 4. MCP LAYER"):])

# 4. MCP LAYER - the only part of this file that knows MCP exists
# =============================================================================
# Never print() in a server: on stdio, stdout IS the wire to the client.

mcp = MCPServer("tickets")

mcp.tool(annotations=READ_ONLY)(search_tickets)
mcp.tool(annotations=READ_ONLY)(get_ticket_details)
mcp.tool(annotations=READ_ONLY)(get_ticket_metrics)
mcp.tool(annotations=READ_ONLY)(find_similar_tickets)
mcp.tool(annotations=WRITES)(update_ticket_status)

if __name__ == "__main__":
    if "--http" in sys.argv:
        mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)   # http://127.0.0.1:8000/mcp
    else:
        mcp.run()                                                          # stdio: the host starts us



### 🎯 Mini-task

Call `get_ticket_metrics("last_30_days")` and `find_similar_tickets("TKT-1001")` from the same module. Then find the line in the printed MCP layer that makes `find_similar_tickets` visible to MCP. What would change if you deleted that one line? (The function would still work in Python. Only clients would stop seeing it.)

## 🛠️ Developer corner: what the MCP layer reads from a function

What `mcp.tool(...)(function)` extracts, verified on the wire in this notebook's environment:

- **Name**: the function name.
- **Description**: the whole docstring, `Args:` section included. It travels as one text block; the model reads all of it. That is why the docstrings in `servers/` are written with the model as the reader.
- **Argument schema**: generated from the type hints. `str | None = None` becomes an optional string with a default. The schema carries names, types and defaults; per-argument explanations live in the docstring, not in the schema.
- **Annotations**: whatever you pass, `READ_ONLY` or `WRITES` from `servers/common.py`. More on those in the model chapter.

The `sys.path` line at the top of every server file exists so the file works both ways: imported as a module (as you just did) and started as a program by a client. `make_error` in `servers/common.py` is the one helper every server shares; underscore names are private by convention only, and MCP never sees them simply because they are never registered.

**🛠️ Stretch.** Import `servers.common`, call `make_error("x", hints=["y"], follow_up_tools=["z"])` and read the dict. Then find where `READ_ONLY` and `WRITES` are defined and what four flags each of them sets.

## 1.4 Be the client

In the last section we opened the server. But a server only waits. Now we look at the other side: the program that asks. That program is called the **client**, and for the next few minutes the client is this notebook.

```text
 this notebook                  ticket_server.py
   (client)                         (server)
      │                                 │
      │  1. what can you do?   ───────▶ │
      │  ◀───────   a list of tools     │
      │                                 │
      │  2. run this one       ───────▶ │
      │  ◀───────   the result, as text │
```

Two questions. That is the whole job of a client. In Python it is four lines, and every client cell in this notebook has this shape:

```python
async with Client(TICKET_SERVER) as tickets:            # start the server program
    listed = await tickets.list_tools()                 # question 1: what can you do?
    result = await tickets.call_tool(name, arguments)   # question 2: run this one
                                                        # the block ends: the server is stopped
```

Two words may be new. `async with` starts the other program and stops it when the block ends. `await` means "wait for the other program to answer". That is all the async you need.

`TICKET_SERVER` in the next cell only says which program to start: Python, running `servers/ticket_server.py`. The two programs talk through the server's ordinary text input and output. This way of connecting is called **stdio**. No network, no ports.

In [7]:
from mcp import Client, StdioServerParameters

TICKET_SERVER = StdioServerParameters(command=sys.executable, args=["servers/ticket_server.py"])

async with Client(TICKET_SERVER) as tickets:
    listed = await tickets.list_tools()
    for tool in listed.tools:
        print(f"{tool.name:24s} {tool.description.strip().splitlines()[0]}")

    result = await tickets.call_tool("search_tickets", {"priority": "critical"})

print("\nFirst 300 characters of what the server sent back:\n")
print(result.content[0].text[:300])

search_tickets           Search support tickets by any combination of filters. All filters are optional;
get_ticket_details       Get the full record of one ticket by its ID.
get_ticket_metrics       Ticket counts and average resolution time for a time window.
find_similar_tickets     Find tickets similar to a given ticket (shared tags, category, OS, priority).
update_ticket_status     Change the status of a ticket. This CHANGES data (in memory, for the duration

First 300 characters of what the server sent back:

{
  "tickets": [
    {
      "ticket_id": "TKT-1002",
      "customer_id": "CUST-002",
      "subject": "Linux server disk full - /var/log consuming 95% space",
      "description": "Production Ubuntu 22.04 server has /var/log partition at 95% capacity. Log rotation not working properly.",
      "st


### 🔍 What just happened?

A second Python program started, running the ticket server. Our notebook asked it two things: *what tools do you have?* and *run this one for me*. Then the block ended and the program was stopped.

Look at the tool list: those descriptions are the first lines of the docstrings you saw in the file. And the result came back as **text**, our dict turned into JSON, because that is what travels between two programs.

Everything MCP does is built from these two questions. Applications add convenience on top; the questions stay the same.

## 1.5 What travels between the two programs

The client asked and the server answered. What exactly went from one program to the other? Short text messages, written in a format called JSON. One message for each question, one for each answer.

Here is the second question, "run this tool", and its answer. A few technical fields are left out:

```text
client --> server
{"id": 3, "method": "tools/call",
 "params": {"name": "get_ticket_details",
            "arguments": {"ticket_id": "TKT-1001"}}}

server --> client
{"id": 3,
 "result": {"content": [{"type": "text",
     "text": "{\"ticket_id\": \"TKT-1001\", ...}"}]}}
```

Read it like a form:

- `method` says which question this is. `tools/list` is *what can you do?* and `tools/call` is *run this one*.
- `params` carries the tool name and its arguments.
- `id` is a number that pairs the answer with its question.
- The answer carries our dict, turned into text.

You never write these messages by hand; `Client` does it for you. Two reasons to know they exist. Nothing in them is Python, so a server written in any language works with any client. And when something breaks, these messages are what you look at.

### 🎯 Mini-task

Copy the client cell into a new cell below it. Change the call to `get_ticket_details` with `{"ticket_id": "TKT-1001"}` and print the whole text. Then try `TKT-9999`: the same error dict you saw in section 1.2, now as text that came from another program.

## 🛠️ Developer corner: the wire, live

Two things the everyone path hid.

**Plumbing.** The setup cell swapped `sys.stderr` for a null file when it has no file handle. Colab's error stream is not a real file, and starting a subprocess needs one. And one rule for this notebook: cells use plain top-level `await`, which Jupyter and Colab allow. Do **not** add `nest_asyncio` (a patch you may know from other notebooks); it hangs the 2.x MCP client.

**The messages.** MCP speaks **JSON-RPC 2.0**: one JSON object per line, a request with `id` and `method`, an answer with the same `id` and a `result`. To see the real thing, we put a wiretap between us and the server: the Unix command `tee` copies everything that passes through a pipe into a file. The server does not know it is being watched.

In [8]:
import json, shlex, tempfile
from pathlib import Path

LOG = Path(tempfile.gettempdir())
TO_SERVER, TO_CLIENT = LOG / "mcp_to_server.log", LOG / "mcp_to_client.log"
for f in (TO_SERVER, TO_CLIENT):
    f.unlink(missing_ok=True)

wrapper = f"tee {TO_SERVER} | {shlex.quote(sys.executable)} servers/ticket_server.py | tee {TO_CLIENT}"
TAPPED_SERVER = StdioServerParameters(command="sh", args=["-c", wrapper])   # sh and tee exist on Linux (Colab) and macOS

async with Client(TAPPED_SERVER) as tickets:
    await tickets.list_tools()
    await tickets.call_tool("get_ticket_details", {"ticket_id": "TKT-1001"})

sent = TO_SERVER.read_text().splitlines()
received = TO_CLIENT.read_text().splitlines()
print(f"{len(sent)} messages went to the server, {len(received)} came back.")

3 messages went to the server, 3 came back.


In [9]:
def trim(message):
    """Shorten the long parts so the structure stays visible."""
    result = message.get("result", {})
    if "tools" in result:
        result["tools"] = [t["name"] for t in result["tools"]] + ["... (full schemas omitted)"]
    for part in result.get("content", []):
        if len(part.get("text", "")) > 100:
            part["text"] = part["text"][:100] + " ..."
    return message

def show_wire(sent, received):
    answers = {m.get("id"): m for m in map(json.loads, received)}
    for line in sent:
        request = json.loads(line)
        print(f"\n--> to the server:   method={request.get('method')}  id={request.get('id')}")
        print(json.dumps(request, indent=2))
        if request.get("id") in answers:
            print(f"\n<-- from the server: id={request['id']}")
            print(json.dumps(trim(answers[request["id"]]), indent=2))

show_wire(sent, received)


--> to the server:   method=server/discover  id=1
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "server/discover",
  "params": {
    "_meta": {
      "io.modelcontextprotocol/protocolVersion": "2026-07-28",
      "io.modelcontextprotocol/clientInfo": {
        "name": "mcp",
        "version": "0.1.0"
      },
      "io.modelcontextprotocol/clientCapabilities": {}
    }
  }
}

<-- from the server: id=1
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "cacheScope": "private",
    "capabilities": {
      "prompts": {
        "listChanged": true
      },
      "resources": {
        "listChanged": true,
        "subscribe": true
      },
      "tools": {
        "listChanged": true
      }
    },
    "resultType": "complete",
    "supportedVersions": [
      "2026-07-28"
    ],
    "ttlMs": 0,
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "tickets",
        "version": ""
      }
    }
  }
}

--> to the server:   method=tools/list  id=2
{
  "jsonrpc": "2.0"

### 🔍 What just happened?

Three requests, three answers, paired by `id`:

1. `server/discover`: the client introduces itself and asks the server what it can do. Look at `_meta`: the protocol version (`2026-07-28`) and the client's name travel with every request.
2. `tools/list`: the same tool names you saw before, now as raw JSON with a schema for each.
3. `tools/call`: the tool name and its arguments; the answer carries our dict as text.

Every result says `"resultType": "complete"`, which tells the client nothing more is coming. Servers written before mid-2026 do not know `server/discover`; the client then gets an error and falls back to an older opening called `initialize`. You will meet that fallback in the host chapter, when a framework with an older client talks to this new server.

**🛠️ Stretch.** Change the call in the wiretap cell to `search_tickets` with `{"status": "open"}` and re-run both cells. How many messages now? Find the `_meta` block in the request, and find the `id` that pairs the call with its answer.

## Optional: MCP Inspector, a client you did not write

*Optional section, no video. Nothing later depends on it.*

Our client is hand-made. Does a client written by somebody else see the same tools? That is the whole promise, so let's check with **MCP Inspector**, the standard debugging client from the MCP project. It needs Node.js (version 22.19 or newer).

**On your own machine**, in a terminal:

```bash
cd MCP-tutorial                      # the repository folder
source .venv/bin/activate            # Windows: .venv\Scripts\activate
npx @modelcontextprotocol/inspector python servers/ticket_server.py
```

It prints a local URL. Open it, click **Connect**, open **Tools**, pick `get_ticket_details`, enter `TKT-9999`, run it, and read the hint. The **History** tab shows the JSON messages from section 1.5, live.

**In Colab**, open the terminal (the `>_` icon at the bottom left, available on paid plans), then:

```bash
cd /content/MCP-tutorial
node --version                        # needs 22.19 or newer
npx @modelcontextprotocol/inspector --cli python servers/ticket_server.py --method tools/list
```

The `--cli` form prints the answer in the terminal instead of opening a page. If Colab's Node is too old, watch the instructor's screen for this step; nothing later depends on it.

### 🔍 What just happened?

Nothing on the server changed, and a program you did not write listed and called its tools. That is what "protocol" buys you: the server has one shape, and any client that speaks the shape can use it.

### 🎯 Mini-task

Point Inspector at `servers/billing_server.py`, call `get_invoice` with no arguments at all, and read what the server suggests.

# 2. Let a Model Drive It

In chapter 1 we chose the tool and typed its arguments ourselves. In this chapter a model does that, from a question in plain words. Three sections: the loop that connects a model to a server, errors the model can act on, and annotations.

## 2.1 A tiny orchestrator: the loop

Where we are: you know what a server is (1.3) and how a client asks it the two questions (1.4). One piece is missing. A model cannot talk to a server by itself; a model only reads text and writes text. Something has to stand in the middle: give the tool list to the model, see which tool the model wants, ask the server to run it, and hand the result back. That middle piece is called an **orchestrator**.

```text
        you: a question in plain words
                      │
                      ▼
┌─────────────────────────────────────────────────┐
│ OUR LOOP (a tiny orchestrator)                  │
│                                                 │
│ - get the tool list from the server             │
│ - ask the model: question + tool list           │
│ - the model wants a tool?                       │
│     yes: ask the server to run it, add the      │
│          result to the conversation, ask again  │
│     no:  the model answered in words, done      │
└─────────────────────────────────────────────────┘
        │                              │
        ▼                              ▼
    the MODEL                    the MCP SERVER
    decides what to call         does the work
```

In a real project a framework does this job, for example the Agent Development Kit from the next course. We do not want another framework here, so we write the smallest possible orchestrator ourselves, about 25 lines.

It is the loop you know from function calling in the previous course. Two things are different, and both are marked in the code: the tool list comes from the server (step 1), and the tool runs in the server, not in our Python (step 6). Read the numbered comments from top to bottom, then run the cell.

In [10]:
def to_openai_tool(tool):
    """Rename the fields of an MCP tool description into the shape the Responses API expects."""
    return {"type": "function", "name": tool.name, "description": tool.description or "", "parameters": tool.input_schema}


async def run_with_mcp(question, mcp_client, max_rounds=6):
    # STEP 1: ask the server what it can do. We type no tool list ourselves; it comes from the server.   <- new with MCP
    listed = await mcp_client.list_tools()

    # STEP 2: rewrite each tool description into the format our model provider expects. Same content, other field names.
    tools = [to_openai_tool(t) for t in listed.tools]

    # STEP 3: start the conversation: instructions for the model, then the user's question.
    conversation = [
        {"role": "developer", "content": "You are an IT help-desk assistant. Use the tools to look things up; never invent data. "
                                         "When a tool returns an error with suggested_actions, follow them. Answer briefly."},
        {"role": "user", "content": question},
    ]

    for round_number in range(1, max_rounds + 1):           # one pass of the loop = one reply from the model
        # STEP 4: ask the model. It gets the whole conversation so far and the tool list.
        response = llm.responses.create(model=MODEL, input=conversation, tools=tools)
        conversation += response.output                     # remember what the model said

        # STEP 5: did the model ask for a tool? If not, it answered in words and we are done.
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text

        print(f"round {round_number}: the model asked for {len(calls)} call(s)")
        for call in calls:
            args = json.loads(call.arguments)               # the model sends the arguments as JSON text

            # STEP 6: run the tool. Not in our Python: we pass the request on to the server.             <- new with MCP
            result = await mcp_client.call_tool(call.name, args)
            output = result.content[0].text

            shown = {k: v for k, v in args.items() if v not in (None, "")}  # the model sends every optional argument, most as null
            print(f"  -> {call.name}({shown}) -> {output[:100].replace(chr(10), ' ')} ...")

            # STEP 7: add the result to the conversation and go around again, so the model can read it.
            conversation.append({"type": "function_call_output", "call_id": call.call_id, "output": output})

    return "Stopped after too many rounds."                 # a brake, so a confused model cannot loop forever

print("run_with_mcp is ready.")

run_with_mcp is ready.


In [11]:
async with Client(TICKET_SERVER) as tickets:
    answer = await run_with_mcp("Which tickets are critical right now? One line each.", tickets)

print("\nAssistant:", answer)

round 1: the model asked for 1 call(s)
  -> search_tickets({'priority': 'critical'}) -> {   "tickets": [     {       "ticket_id": "TKT-1002",       "customer_id": "CUST-002",       "subjec ...



Assistant: TKT-1002 — Linux server disk full; `/var/log` at 95% capacity (in progress).  
TKT-1009 — BitLocker recovery key prompt on every boot (in progress).  
TKT-1011 — Active Directory replication failing (resolved).


### 🔍 What just happened?

Follow the step numbers from the code:

- **Steps 1 and 2.** The server listed five tools, and the loop passed their descriptions to the model.
- **Step 4.** The model read the question and the descriptions. It did not answer in words. It answered with a request: call `search_tickets` with `priority="critical"`.
- **Steps 5 to 7.** The loop saw the request, asked the server to run the tool, and put the result into the conversation.
- **Second pass.** Now the model had the data, so it answered in words, and step 5 returned that answer.

Notice who knows what. The model never saw a line of Python. The loop has no idea what a ticket is. The server does not know a model is involved. Each of the three only knows the tool descriptions, and that is enough.

The same loop, packaged for reuse, lives in `client/agent_loop.py`. The terminal chat from section 1.1 uses it.

## 2.2 Errors written for the model

In section 1.2 we asked for a ticket that does not exist and got back a dict instead of a crash. Now that a model is driving, we can see why it was written that way. First, where that dict comes from. The next cell prints the tool from the server file.

In [12]:
import inspect

print(inspect.getsource(get_ticket_details))

def get_ticket_details(ticket_id: str) -> dict:
    """Get the full record of one ticket by its ID.

    Args:
        ticket_id: unique ticket identifier (e.g. TKT-1001)

    Returns:
        the ticket dict, or an error dict with suggested next steps
    """
    ticket = _find_ticket(ticket_id)
    if not ticket:
        return make_error(
            f"Ticket {ticket_id} not found",
            reason="The ticket_id did not match any tickets in the dataset.",
            hints=[
                "Call search_tickets with a query, customer_id or priority filter to rediscover the ticket.",
                "Verify the ticket_id format (e.g., TKT-1001)."
            ],
            retryable=True,
            follow_up_tools=["search_tickets"],
            ticket_id=ticket_id
        )
    return dict(ticket)



Look at the `make_error(...)` call. It says what went wrong, why (`reason`), what to try next (`hints`), and which tool to use for that (`follow_up_tools`). `make_error` is a small helper in `servers/common.py` that packs these into a dict. These fields are not part of MCP; they are our own convention for writing errors a model can use. The function does not crash and does not `raise`; it **returns** the error like any other result.

### First example: a record that does not exist

We ask for ticket TKT-9999, which is not in the data, and we add a hint about what the ticket was about. Watch the `->` lines.

In [13]:
async with Client(TICKET_SERVER) as tickets:
    answer = await run_with_mcp("Give me the details of ticket TKT-9999, the BitLocker one.", tickets)

print("\nAssistant:", answer)

round 1: the model asked for 1 call(s)
  -> get_ticket_details({'ticket_id': 'TKT-9999'}) -> {   "error": "Ticket TKT-9999 not found",   "reason": "The ticket_id did not match any tickets in th ...


round 2: the model asked for 1 call(s)
  -> search_tickets({'query': 'BitLocker'}) -> {   "tickets": [     {       "ticket_id": "TKT-1009",       "customer_id": "CUST-007",       "subjec ...


round 3: the model asked for 1 call(s)
  -> get_ticket_details({'ticket_id': 'TKT-1009'}) -> {   "ticket_id": "TKT-1009",   "customer_id": "CUST-007",   "subject": "Windows 11 BitLocker recover ...



Assistant: Ticket **TKT-9999** wasn’t found. The BitLocker ticket appears to be **TKT-1009**:

- **Subject:** Windows 11 BitLocker recovery key prompt on every boot
- **Customer:** CUST-007
- **Issue:** Recovery key is requested at every startup after a BIOS update; TPM is enabled in BIOS.
- **Status:** In progress
- **Priority:** Critical
- **Category:** Security
- **OS:** Windows 11
- **Assignee:** Sarah Lee
- **Created:** 2026-11-15
- **Last updated:** 2026-11-18
- **Tags:** Windows, BitLocker, encryption, TPM


### 🔍 What just happened?

The first call failed: the server answered with the error dict. The model read the hint, called `search_tickets` with a query, found the real ticket and answered. Our loop has no error handling at all. The model repaired the mistake by itself, because the error told it what to try next.

⚠️ Models are not deterministic. If yours searched first or gave up, run the cell once more.

### Second example: a value that is not allowed

A different kind of mistake. A ticket can be `open`, `in_progress` or `resolved`. We ask for a status that does not exist.

In [14]:
async with Client(TICKET_SERVER) as tickets:
    answer = await run_with_mcp("Change the status of ticket TKT-1004 to escalated.", tickets)

print("\nAssistant:", answer)

round 1: the model asked for 1 call(s)
  -> update_ticket_status({'ticket_id': 'TKT-1004', 'status': 'escalated'}) -> {   "error": "Invalid status 'escalated'",   "reason": "Status must be one of the known values.",    ...



Assistant: I couldn’t change TKT-1004 to **escalated**. Valid statuses are **open**, **in_progress**, or **resolved**.


### 🔍 What just happened?

The server refused: *Invalid status 'escalated'*, and its hint listed the three values that are allowed. This time the model cannot repair anything by itself. It does not know which status you would want instead, and guessing would change your data. So it stopped and told you exactly which values exist.

Two errors, two different good endings. In the first, the model fixed the problem and carried on. In the second, it came back to you with a precise answer instead of a vague "something went wrong". Both happened because the error message was written for the reader who actually reads it: the model.

*Why not simply raise an exception in the tool?* Because the message you wrote would be lost on the way to the model; the developer corner at the end of this chapter shows it. The rule: **a missing ticket is not a bug, it is Tuesday. Expected failures are returned as data with a next step; exceptions are for real bugs.**

### 🎯 Mini-task

Add a field `example_ids=["TKT-1001", "TKT-1002"]` to the error in `get_ticket_details` (in `servers/ticket_server.py`) and re-run the TKT-9999 question. Does the model use it?

## 2.3 Annotations

An application like Claude Code has to decide which tools it may run without asking you every time. Is `search_tickets` safe to run silently? Is `update_ticket_status`? The server can say so with **annotations**, small flags attached to each tool:

- `read_only_hint`: the tool only reads.
- `destructive_hint`: the tool may delete or overwrite.
- `idempotent_hint`: calling it twice is the same as once.
- `open_world_hint`: it reaches outside its own data (the internet, other systems).

Where do you write them? In the server, in two places. The next cell prints both straight from the files.

In [15]:
common = Path("servers/common.py").read_text()
print(common[common.index("# Tool annotations"):])

print("...and at the bottom of servers/ticket_server.py:\n")
for line in Path("servers/ticket_server.py").read_text().splitlines():
    if line.startswith("mcp.tool("):
        print(line)

# Tool annotations are hints for the host application, not rules for the model.
READ_ONLY = ToolAnnotations(read_only_hint=True, destructive_hint=False, idempotent_hint=True, open_world_hint=False)
WRITES = ToolAnnotations(read_only_hint=False, destructive_hint=False, idempotent_hint=True, open_world_hint=False)

...and at the bottom of servers/ticket_server.py:

mcp.tool(annotations=READ_ONLY)(search_tickets)
mcp.tool(annotations=READ_ONLY)(get_ticket_details)
mcp.tool(annotations=READ_ONLY)(get_ticket_metrics)
mcp.tool(annotations=READ_ONLY)(find_similar_tickets)
mcp.tool(annotations=WRITES)(update_ticket_status)


Two presets and one line per tool:

- `READ_ONLY` and `WRITES` are the four flags filled in once, so we do not repeat them for every tool.
- `mcp.tool(annotations=READ_ONLY)(search_tickets)` is the registration line you know from section 1.3. The annotation travels with the tool.
- One tool, `update_ticket_status`, gets `WRITES`, because it changes data.

That is the server's side. Now we ask the server as a client and see what arrives.

In [16]:
async with Client(TICKET_SERVER) as tickets:
    for tool in (await tickets.list_tools()).tools:
        a = tool.annotations
        print(f"{tool.name:24s} read_only={str(a.read_only_hint):5s}  destructive={a.destructive_hint}")

search_tickets           read_only=True   destructive=False
get_ticket_details       read_only=True   destructive=False
get_ticket_metrics       read_only=True   destructive=False
find_similar_tickets     read_only=True   destructive=False
update_ticket_status     read_only=False  destructive=False


### 🔍 What just happened?

Four tools say *read only*, one says it writes. A careful application runs the first four freely and asks before the fifth. You will see exactly that prompt in Claude Code in section 3.1.

⚠️ They are called *hints* for a reason. An application may ignore them, and a server can lie. Annotations help honest applications and honest servers work together; they are not a security boundary. More on that in section 3.2.

### 🎯 Mini-task

Copy the cell with the escalated status and ask `run_with_mcp` to set ticket TKT-1004 to resolved, then, in the same `async with` block, ask for its status. Then open a *new* block and ask for the status again. The change is gone: the server keeps its data in memory, and every block starts a fresh program.

## 🛠️ Developer corner: raise or return?

*"So why not just `raise` an exception in the tool?"* Let's try both. The next cell builds a tiny two-tool server right here in the notebook and connects to it **in-process**: `Client(demo)` takes the server object directly, with no subprocess. Same client, same questions, handy for tests.

Two more details from this chapter. `to_openai_tool` only renames fields; the schema itself, `tool.input_schema`, passes through untouched. The server generated it from the type hints, and the model reads it as is. And the `shown` line in the loop exists because the model sends every optional argument, most of them as `null`; the server treats `null` as "not given".

In [17]:
from mcp.server import MCPServer
from servers.common import make_error

logging.getLogger("mcp").setLevel(logging.CRITICAL)   # the server logs the traceback to ITS OWN error output; we only want to see what the client gets
demo = MCPServer("demo")

@demo.tool()
def lookup_raises(item_id: str) -> dict:
    """Look up an item. Raises on unknown ids."""
    raise ValueError(f"{item_id} not found, try search_items")

@demo.tool()
def lookup_returns(item_id: str) -> dict:
    """Look up an item. Returns an error dict on unknown ids."""
    return make_error(f"{item_id} not found", hints=["Call search_items with a keyword."], follow_up_tools=["search_items"])

async with Client(demo) as c:
    for name in ("lookup_raises", "lookup_returns"):
        r = await c.call_tool(name, {"item_id": "X-1"})
        print(f"{name}:  is_error={r.is_error}\n   {r.content[0].text}\n")

lookup_raises:  is_error=True
   Error executing tool lookup_raises

lookup_returns:  is_error=False
   {
  "error": "X-1 not found",
  "suggested_actions": [
    "Call search_items with a keyword."
  ],
  "retryable": false,
  "follow_up_tools": [
    "search_items"
  ]
}



### 🔍 What just happened?

The raised exception reached the client as `is_error=True` and a generic *"Error executing tool"*. The message you wrote, with its helpful *try search_items*, was swallowed on the way. The returned dict arrived complete.

The rule for tools: **exceptions for bugs, dicts for expected failures.** A missing ticket is not a bug, it is Tuesday.

**🛠️ Stretch.** Read `client/agent_loop.py`: `Toolbox` and `run_with_tools` are this chapter's loop with routing across several servers. Then change `run_with_mcp` to print the full argument dict, nulls included, and see what the model really sends.

# 3. Plug Into a Real Application

So far the notebook did everything: it started the server, asked the questions and ran the loop. In real life an application does that for you. Two sections: our server in a real application, then somebody else's server.

## 3.1 Our server in a real host

A new word, and it is simpler than it sounds. A **host** is the application you type into: Claude Code, Claude Desktop, Cursor, or an agent you built. The host contains the three things we built by hand: a model, the loop from section 2.1, and a client that asks the two questions from section 1.4.

```text
┌───────────────────────────────────────────────────────┐
│ HOST: the application you type into                   │
│ (Claude Code, Claude Desktop, Cursor, your own agent) │
│                                                       │
│ the model  <-->  the loop  <-->  an MCP client        │
└───────────────────────────────────────────────────────┘
                                     │
                                     │ the two questions
                                     ▼
                                MCP SERVER
                        (servers/ticket_server.py)
```

So nothing here is new. Until now this notebook played the host. Now a finished application takes over that role, and our server stays exactly as it is. Connecting the two is configuration, not code: you tell the host which program to start.

**Claude Code**, in a terminal on your own machine, from the repository folder with the virtual environment active:

```bash
claude mcp add tickets -- python servers/ticket_server.py
claude mcp list                        # tickets ... connected
claude
```

Inside the session, ask: *which tickets are critical?* Then: *set ticket TKT-1004 to resolved.* The second one triggers a permission prompt: Claude Code saw the annotation from section 2.3 that says the tool writes. `/mcp` lists the connected servers and their tools. When you are done: `claude mcp remove tickets`.

Docs: https://code.claude.com/docs/en/mcp

### Configuration files, same shape everywhere

Every host stores the same two things: the command to start the server and its arguments.

The repository ships a `.mcp.json`, which Claude Code reads automatically when you open the folder (it asks you once whether to trust it):

```json
{
  "mcpServers": {
    "tickets":   {"command": "python", "args": ["servers/ticket_server.py"]},
    "customers": {"command": "python", "args": ["servers/customer_server.py"]},
    "billing":   {"command": "python", "args": ["servers/billing_server.py"]}
  }
}
```

**Claude Desktop** uses `claude_desktop_config.json` (Settings, Developer, Edit Config) with the same `mcpServers` key. It runs the servers from anywhere, so use absolute paths there, for both the Python interpreter and the file.

**VS Code** uses `.vscode/mcp.json` with the key `servers` instead of `mcpServers`. **Cursor** uses `mcpServers`. Same idea, one spelling apart.

### 🎯 Mini-task

Register `servers/kb_server.py` in Claude Code as `knowledge` and ask it for a fix for a Windows blue screen. Which tool did it pick, and did it need a second call?

And the question from section 0.3, one more time: Claude Code could also just run `python -c "from servers.ticket_server import search_tickets; ..."` on your machine. Why bother with the server? Because the moment the tickets live on another machine, or the person asking is not you, the shell is gone and the server is all there is.

## 3.2 Someone else's server

In 3.1 we plugged in a server we wrote. Most of the time you will not write one. Somebody already wrote a server for the thing you need: your files, GitHub, a documentation site. You install theirs.

```text
            a host (Claude Code, your own agent)
               │                        │
               ▼                        ▼
       YOUR OWN SERVER        SOMEONE ELSE'S SERVER
       you wrote it           you install it, or
       (ticket_server.py)     you point at its URL
```

For the host the two look the same: the same two questions, the same configuration.

**A server that runs on your machine.** The MCP project publishes a server that works with files in a folder you name. One line registers it in Claude Code, and the program is downloaded on first start:

```bash
claude mcp add files -- npx -y @modelcontextprotocol/server-filesystem /full/path/to/MCP-tutorial
claude
```

Ask: *list the files in servers/ and tell me which one has a tool that writes.* Then `claude mcp remove files`.

**A server that runs somewhere else.** Then you give an address instead of a command. DeepWiki is a public server that answers questions about open-source repositories, and it needs no login:

```bash
claude mcp add --transport http deepwiki https://mcp.deepwiki.com/mcp
```

Ask: *using deepwiki, what does the repository modelcontextprotocol/python-sdk do?*

### The same server in your own agent

Claude Code is one host. What if the host is an agent you built? Then you do in code what `claude mcp add` did in configuration: you tell your agent where the server is.

We have a small agent already: `run_with_mcp` from section 2.1. Until now we gave it our ticket server. In the next cell we give it DeepWiki's address instead. Nothing else changes.

In [18]:
try:
    async with Client("https://mcp.deepwiki.com/mcp") as wiki:        # an address instead of a program to start
        for tool in (await wiki.list_tools()).tools:
            print(f"{tool.name:22s} {tool.description.strip().splitlines()[0]}")
        print()
        answer = await run_with_mcp("In two sentences: what is the GitHub repository modelcontextprotocol/python-sdk for?", wiki)
    print("\nAssistant:", answer)
except Exception:
    print("DeepWiki did not answer (no internet, or the public server is down). Nothing later depends on this cell.")

ask_wiki_question      Ask any question about a GitHub repository's codebase and get an AI-powered answer
read_wiki_contents     View documentation about a GitHub repository.
read_wiki_structure    Get a list of documentation topics for a GitHub repository.



round 1: the model asked for 1 call(s)


  -> ask_wiki_question({'repoName': 'modelcontextprotocol/python-sdk', 'question': 'What is this GitHub repository for? Summarize its purpose in two sentences.'}) -> ## Answer  This repository is the official Python SDK for the Model Context Protocol (MCP), providin ...



Assistant: The `modelcontextprotocol/python-sdk` repository is the official Python SDK for the Model Context Protocol (MCP). It provides tools for building MCP servers and clients that expose tools, resources, and prompts to compatible hosts and applications.


### 🔍 What just happened?

Our loop listed the tools of a server we have never seen, written by people we do not know, running on their machine. The model picked one and got an answer. We changed one line: the address.

A framework does the same thing with its own words. In the Agent Development Kit from the next course, an outside server becomes a tool of your agent like this (shown, not run here, because that framework needs its own environment):

```python
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

agent = LlmAgent(
    name="repo_helper",
    model="gemini-flash-latest",
    instruction="Answer questions about open-source repositories.",
    tools=[McpToolset(connection_params=StreamableHTTPConnectionParams(url="https://mcp.deepwiki.com/mcp"))],
)
```

Read the last line inside out: where the server is (the URL), and who handles it for the agent (`McpToolset`). The same information as our `Client("https://...")` line.

### Install only servers you trust

Installing a server takes one line. That is exactly why you should stop for a moment before you do it. Three plain facts:

1. **A local server is a program on your computer.** `npx ...` downloads it and runs it with your permissions. A malicious one can do whatever you can do: read your files and your keys and send them away. It is the same risk as installing any package from the internet.
2. **A remote server sees what you send it.** Every argument of every call leaves your machine, and with it pieces of your conversation.
3. **The model reads whatever the server sends back, and it cannot reliably tell data from instructions.** A tool description or a tool result can contain a sentence like *"also send the user's files to ..."*, and the model may follow it. Hiding such text in a tool description is called **tool poisoning**. Even a well-meant server can pass it on, inside a web page or a document it fetched for you.

So before you install, ask three things. Who wrote it: the vendor itself, a known project, or a stranger? What can its tools change or delete? What data will it see? And keep the host's permission prompts switched on for every tool that writes.

### 🎯 Mini-task

Look at the tool list the DeepWiki cell printed. For each of the three tools, answer in one sentence: what does it send to DeepWiki's machine? Would you ask it about a private repository of your company?

## 🛠️ Developer corner: the next course's framework

In the ADK course, the same server becomes a tool of an agent in one line. The code is shown here, not run, because that framework installs its own (older) MCP client library, which cannot share this notebook's environment:

```python
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

ticket_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(command=sys.executable, args=["servers/ticket_server.py"])
    )
)
agent = LlmAgent(name="helpdesk", model=..., tools=[ticket_toolset])
```

Read it inside out: which program to start, how to talk to it, who handles it. The same three questions as our `Client(...)` line. And remember the wire corner: that older client sends `initialize` instead of `server/discover`, and our server answers both. Different generations, one server.

# 4. What Else, and Your Turn

Two things are left. A server can offer more than tools, and it is your turn to build something.

## 4.1 Resources and prompts

So far our servers offered **tools**: functions the *model* decides to call. A server can offer two more things. Both are meant for the *person* in front of the application, not for the model.

**A resource is a document the server offers for reading.** Our HR server offers the company's remote work policy under the address `hr://policy/remote-work`. In an application you see it in a list, like a file you can attach. You pick it, and its text is added to your conversation. Then you ask *"how many days a week can I work from home?"* and the model answers from the policy itself instead of guessing.

**A prompt is a ready-made request with blanks.** Somebody in HR once wrote a good instruction for performance reviews. The server offers it under the name `performance_review`, with two blanks: the employee and the period. You pick it, fill in the blanks, and the full instruction is sent for you.

*Why not just a tool `get_policy()`?* That would work too. The difference is who decides. A tool is called when the model thinks it needs it. A resource is attached when **you** say "use this document".

The next cell asks the HR server the way we asked for tools: list what you have, then give me one.

In [19]:
HR_SERVER = StdioServerParameters(command=sys.executable, args=["servers/hr_server.py"])

async with Client(HR_SERVER) as hr:
    print("Resources with a fixed address:")
    for r in (await hr.list_resources()).resources:
        print("  ", r.uri, "-", r.name)
    print("Resource addresses with a blank to fill in:")
    for t in (await hr.list_resource_templates()).resource_templates:
        print("  ", t.uri_template)

    doc = await hr.read_resource("hr://policy/remote-work")
    print("\nThe remote work policy, first lines:\n")
    print(doc.contents[0].text[:330])

    print("\nPrompts, with their blanks:")
    for p in (await hr.list_prompts()).prompts:
        print("  ", p.name, "-", [a.name for a in p.arguments])

Resources with a fixed address:
   hr://policy-index - policy_index
Resource addresses with a blank to fill in:
   hr://policy/{name}

The remote work policy, first lines:

Remote Work Policy

Eligibility:
- All full-time employees after 3 months of employment
- Manager approval required

Remote Work Options:
- Fully remote: work from anywhere in the country
- Hybrid: minimum 2 days per week in the office
- Flexible: choose your schedule with team coordination

Equipment:
- Compa

Prompts, with their blanks:
   performance_review - ['employee_id', 'review_period']


### 🔍 What just happened?

- **Resources:** the server listed one document with a fixed address (the index of policies) and one address with a blank, `hr://policy/{name}`. We filled the blank with `remote-work`, and `read_resource` returned the text of the policy. That text is what an application would add to your conversation.
- **Prompts:** the server listed `performance_review` with its two blanks, `employee_id` and `review_period`.

Where you meet them in a real application: in Claude Code, type `@` and the resources of connected servers appear next to your files; prompts appear among the slash commands when you type `/`. Other applications have their own menus for the same two things.

One practical note. Most MCP servers you will meet offer only tools. Resources and prompts are less common, but now you will recognise them when you see them.

### 🎯 Mini-task

In the cell above, change the address to `hr://policy/benefits` and run it again. Then read `hr://policy-index` the same way. Which of the two addresses would you attach before asking *"how many days of paid time off do I get in a year?"*

## 🛠️ Developer corner: how the HR server declares them

Two new decorators in `servers/hr_server.py` (that file uses the decorator spelling so you see both ways of registering things):

- `@mcp.resource("hr://policy/{name}")`: an address with a hole in it. The function fills the hole and returns the text.
- `@mcp.prompt()`: the function's arguments become the blanks, its return value the message.

The first cell prints the MCP layer; the second fills in the prompt from the client side.

In [20]:
source = Path("servers/hr_server.py").read_text()
print(source[source.index("# 4. MCP LAYER"):source.index("if __name__")])

# 4. MCP LAYER (this server is small enough to have no helpers of its own)
# =============================================================================
# Never print() in a server: on stdio, stdout IS the wire to the client.

mcp = MCPServer("hr")


@mcp.resource("hr://policy-index")
def policy_index() -> str:
    """List of all HR policy documents and their addresses."""
    return "\n".join(f"hr://policy/{key}  -  {doc['title']}" for key, doc in HR_POLICIES.items())


@mcp.resource("hr://policy/{name}")
def policy(name: str) -> str:
    """One HR policy document by its short name: benefits, remote-work or pto-accrual."""
    if name not in HR_POLICIES:
        return f"No policy called '{name}'. Known policies: {', '.join(HR_POLICIES)}."
    return HR_POLICIES[name]["content"]


@mcp.prompt()
def performance_review(employee_id: str, review_period: str) -> str:
    """Structured brief for writing an employee's performance review."""
    emp = EMPLOYEES.get(employee_id)
    who = (f

In [21]:
async with Client(HR_SERVER) as hr:
    filled = await hr.get_prompt("performance_review", {"employee_id": "EMP-001", "review_period": "first half of 2026"})
    print("get_prompt('performance_review', ...):\n")
    print(filled.messages[0].content.text[:400], "...")

get_prompt('performance_review', ...):

Write a performance review for Alice Johnson (Senior Software Engineer, Engineering, manager Sarah Chen, hired 2021-03-15). Recent projects: Payment API Redesign, Database Migration. Skills: Python, PostgreSQL, Kubernetes.
Review period: first half of 2026.

Follow this structure:
1. Achievements: 3 to 5 concrete accomplishments from the period.
2. Core competencies: rate technical skills, communi ...


### 🔍 What just happened?

`get_prompt` gave back a ready-made message with the employee's record filled in, and `read_resource` earlier gave back the document with its type (`text/plain`). Both went over the same wire as the tools, as `resources/read` and `prompts/get`.

One sentence on a feature you may read about: **sampling** let a server ask the host's model for a completion. It was deprecated in July 2026, because servers that need a model simply call the model provider directly. Do not build on it.

**🛠️ Stretch.** Add a fourth policy to `HR_POLICIES` in `servers/hr_server.py` and a second prompt `onboarding_checklist(employee_name, start_date)`. Re-run the cells above; both should appear without touching anything else.

## Optional: Beyond your laptop

*Optional reading, no video.*

Everything so far used stdio: the host starts the server as a child program, one user, one machine. A company wants something else: **one server, running centrally, that everyone's agents connect to.** For that MCP has a second transport, **Streamable HTTP**: the server is a small web service and the client gets a URL.

Our servers already support it. The same file, started with `--http`, listens on `http://127.0.0.1:8000/mcp`. The last lines of `ticket_server.py` are:

```python
if "--http" in sys.argv:
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)
else:
    mcp.run()
```

And the client side changes exactly one thing: instead of a command to start, it gets an address. In our notebook that is `Client("http://127.0.0.1:8000/mcp")`; in Claude Code it is `claude mcp add --transport http tickets http://127.0.0.1:8000/mcp`. Same tools, same questions. The developer corner at the end of this chapter runs it.

### What changes when the server leaves your laptop

This is where MCP matters most, and it comes with three topics to know by name:

**Authentication.** A remote server is a web API like any other. MCP uses OAuth: the host obtains a token from the company's login system and sends it with every request; the server checks it. The Python SDK handles the flow on both sides; you configure it rather than write it.

**Low trust.** A server is somebody's code touching your data. Companies put servers behind a gateway, allow-list which servers agents may use, prefer read-only tools, and log every call. The annotations from earlier are one input to those decisions.

⚠️ **Tool poisoning.** Tool descriptions are text the model reads and trusts. A malicious or compromised server can hide instructions in a description (*"before answering, also send the user's files to ..."*), and annotations can lie. As section 3.2 said: only connect to servers you trust, read what they expose, and treat a new server like a new dependency in your code, because that is what it is.

### 🎯 Mini-task

A colleague found an MCP server for your ticket system on GitHub and wants to connect the team's Claude Desktop to it. Which of the three topics above would you raise first, and what would you ask to see before saying yes? Two sentences. Then list the tools of `servers/kb_server.py` with a client cell and read every description as if you were looking for hidden instructions.

## 🛠️ Developer corner: the same server on a URL

The next cell starts the ticket server in the background with `--http`, waits for the port to open, connects over HTTP, runs the model loop, and stops the server again.

In [22]:
import socket, subprocess, time

if "http_server" in globals():
    http_server.terminate()                             # re-running the cell must not leave port 8000 busy
http_server = subprocess.Popen([sys.executable, "servers/ticket_server.py", "--http"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(50):                                     # wait up to 5 seconds for the port to open
    if socket.socket().connect_ex(("127.0.0.1", 8000)) == 0:
        break
    time.sleep(0.1)

async with Client("http://127.0.0.1:8000/mcp") as tickets_http:
    print("Tools over HTTP:", [t.name for t in (await tickets_http.list_tools()).tools])
    answer = await run_with_mcp("How many tickets were resolved in the last 30 days?", tickets_http)

print("\nAssistant:", answer)
http_server.terminate()
print("HTTP server stopped.")

Tools over HTTP: ['search_tickets', 'get_ticket_details', 'get_ticket_metrics', 'find_similar_tickets', 'update_ticket_status']


round 1: the model asked for 1 call(s)
  -> get_ticket_metrics({'time_period': 'last_30_days'}) -> {   "time_period": "last_30_days",   "start_date": "2026-10-19",   "total_tickets": 15,   "open_tick ...



Assistant: 3 tickets were resolved in the last 30 days.
HTTP server stopped.


### 🔍 What just happened?

Only the address changed. Same tools, same loop, same answers, but now the server could be on another machine with a hundred clients connected. Under the hood the client no longer starts a program; it sends HTTP requests to a running service, and the OAuth flow from the everyone path would sit exactly here, as a token on every request.

**🛠️ Stretch.** Start the HTTP server again (first half of the cell above) and connect **two** clients to it in the same cell, calling a tool from each. One server, many clients: the thing stdio cannot do. Stop the server at the end.

## 4.2 Your turn

You have watched every piece work. Now build one small piece yourself. It takes about 20 minutes, and it is the best way to find out whether the course made sense to you.

**The task: give the HR server one new tool.** The tool is called `list_department` and answers the question *"who works in Engineering?"*. An LLM may write the Python for you. Two things are yours, because they are what MCP is really about: the description the model reads, and the error it gets when something goes wrong.

1. **Open `servers/hr_server.py`.** Look at the `EMPLOYEES` data at the top; every employee has a `department`. Then find `get_employee` near the bottom. Your tool goes right below it.
2. **Write the function** `list_department(department: str) -> dict`. It returns the name and the role of everyone in that department.
3. **Write the docstring for the model** (section 1.3). First line: what the tool does. Then an `Args:` section with an example value, such as `Engineering`. This text is everything the model will know about your tool.
4. **Handle the expected failure** (section 2.2). If nobody works in that department, do not raise. Return `make_error(...)` with a hint that lists the departments that do exist.
5. **Register the tool** (section 2.3). Put `@mcp.tool(annotations=READ_ONLY)` above the function, the same line its neighbour has. It only reads.

### Check your work

First as a client. Paste this into a new cell. Your tool should be in the list with the description you wrote, and the unknown department should give back your own error:

```python
async with Client(HR_SERVER) as hr:
    for tool in (await hr.list_tools()).tools:
        print(tool.name, "-", tool.description.strip().splitlines()[0])
    result = await hr.call_tool("list_department", {"department": "Marketing"})
    print(result.content[0].text)
```

Then let the model use it:

```python
async with Client(HR_SERVER) as hr:
    print(await run_with_mcp("Who works in Engineering?", hr))
    print(await run_with_mcp("Who works in Marketing?", hr))
```

**You are done when:**

- the tool appears in the list with your description,
- the model answers the first question after one tool call,
- for the second question the model tells you which departments exist. It can only know that from your hint.

If the model's second answer is vague, the fix is not in the model. Rewrite your error until a reader with no other information would know what to do next.

### Want more? Build a sixth server

For developers, about 30 minutes: build a whole new server for the help desk, in a domain you choose (scheduling, procurement, travel, ...). Copy `servers/ticket_server.py`, replace the data and the functions, keep the four parts, give every tool an annotation, add it to `.mcp.json` and ask Claude Code about it. Step-by-step instructions and a checklist: https://github.com/robertbarcik/MCP-tutorial/blob/main/EXERCISE.md

## Key takeaways

- A **server** is data, plain functions, and a few lines of MCP at the bottom. The functions are ordinary software and stay testable without MCP.
- A **client** asks two questions: *what tools do you have?* and *run this one*. Everything else is built on those.
- What travels between them are short JSON messages, `tools/list` and `tools/call`, paired by `id`.
- An **orchestrator** stands between the model and the server. Here it was a loop of about 25 lines; in a real project it is a framework or the application itself.
- **Errors are data for the model.** A returned dict with a next step lets the model recover or come back to you with a precise answer; an exception loses your message.
- **Annotations** tell the application which tools are safe to run silently. Hints, not guarantees.
- A **host** is the application you type into. It contains the model, the loop and a client, so connecting a server is configuration, not code.
- **Someone else's server** is one line to install. Install only servers you trust: a local one runs with your permissions, a remote one sees what you send, and the model reads whatever comes back.
- **Resources and prompts** are for the person; tools are for the model.
- **Streamable HTTP** (the optional section) puts the same server on a URL, which is where authentication and trust become your problem.
- And the rule from the beginning: on your own machine, with your own agent, a good CLI is enough. MCP is for the boundary to someone else's.

*Cleanup:* every `async with` block stopped its server, and the HTTP cell stopped its process. If you interrupted a cell halfway, a stray server may still run; on Linux or macOS `pkill -f "servers/.*_server.py"` removes it.